In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv('../data/liver_cancer_prediction.csv')
df.head()

,Country,Region,Population,Incidence_Rate,Mortality_Rate,Gender,Age,Alcohol_Consumption,Smoking_Status,Hepatitis_B_Status,...,Herbal_Medicine_Use,Healthcare_Access,Screening_Availability,Treatment_Availability,Liver_Transplant_Access,Ethnicity,Preventive_Care,Survival_Rate,Cost_of_Treatment,Prediction
0,Nigeria,Sub-Saharan Africa,340672131,15.381360,6.160480,Male,81,Low,Smoker,Negative,...,No,Poor,Available,Available,No,Hispanic,Good,17.724793,47486.167423,Yes
1,United Kingdom,Europe,1054632817,3.306101,14.392985,Male,87,Low,Smoker,Negative,...,Yes,Good,Available,Not Available,No,Mixed,Moderate,19.558853,13782.265151,No
2,India,South Asia,751241440,9.325053,12.777878,Male,34,Moderate,Smoker,Negative,...,No,Good,Not Available,Not Available,No,Mixed,Moderate,68.468892,25308.034132,No
3,Colombia,South America,1167333367,9.399658,8.634609,Male,63,Low,Non-Smoker,Positive,...,No,Good,Not Available,Not Available,Yes,Hispanic,Moderate,18.200287,38221.622202,No
4,Iran,Middle East,1082070787,9.665663,12.422518,Male,85,High,Non-Smoker,Positive,...,Yes,Moderate,Available,Available,Yes,Mixed,Moderate,45.019153,26765.301404,No


In [3]:
df_clean = df.dropna()
selected_columns = ['Age', 'Gender', 'Alcohol_Consumption', 'Smoking_Status', 'Hepatitis_B_Status', 
                    'Hepatitis_C_Status', 'Obesity', 'Diabetes', 'Prediction']
df_selected = df_clean[selected_columns]
X = df_selected.drop(columns='Prediction')
y = df_selected['Prediction']
X = pd.get_dummies(X, drop_first=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
pickle.dump(scaler, open('../model/scaler_cancer.pkl', 'wb'))

a1 = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, class_weight='balanced')
a2 = GradientBoostingClassifier(n_estimators=50, max_depth=10, random_state=42)
a3 = LogisticRegression(random_state=42, class_weight='balanced')

ensemble = VotingClassifier(estimators=[('RF', a1), ('GB', a2), ('LR', a3)], voting='soft')
ensemble.fit(X_train_scaled, y_train)
y_pred = ensemble.predict(X_test_scaled)
    
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
pickle.dump(ensemble, open('../model/livercancer_model.pkl', 'wb'))

Accuracy: 74.86%
